In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# 1. Device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치", device)

# 2. 토큰(단어) 사전 만들기
# 숫자 문자열 "1"~"10"만 단어를 사용함
## <PAD>: 길이가 다른 문장을 같은 길이로 맞출때 빈칸처럼 채우는 토큰
## <SOS>: 디코더가 출력을 시작할 때 넣는 시작 토큰
## <EOS>: 문장을 나타내는 토큰

PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2

token_to_idx  = {
    "<PAD>":0,
    "<SOS>":1,
    "<EOS>":2,
    "1":3,
    "2":4,
    "3":5,
    "4":6,
    "5":7,
    "6":8,
    "7":9,
    "8":10,
    "9":11,
    "10":12
}

# index -> token으로 다시 바뀌기 위한 사전
# -> 나중에 예측 결과를 사람이 읽을 수 있게 바꾸는데 사용
idx_to_token = {v:k for k,v in token_to_idx.items()}


#전체 단어의 수
vocab_size = len(idx_to_token)

# ex) 1 2 3
# output 3 2 1

사용 장치 cuda


In [29]:
#Seq2Seq가 학습할 입력-정답 한쌍을 랜덤으로 만드는 함수
#랜덤시퀀스를 생성해서 문자열로 변환했다가 다시 숫자로 전환하는
#인코딩 디코딩을 한번 갔다오는 기본구조
def generate_sample(min_len=3, max_len=5):
    # 시퀀스 길이를 3~5 사이에서 랜덤 선택
    length = random.randint(min_len,max_len)

    # 길이 만큼 1~10 까지의 랜덤 숫자 생성
    seq = [str(random.randint(1,10)) for _ in range(length)]

    # 문자열 토큰을 숫자 인덱스 변환
    src = [token_to_idx[token] for token in seq]

    # 정답(target)은 뒤집은 시퀀스
    tgt = [token_to_idx[token] for token in reversed(seq)]

    return src,tgt

In [30]:
# 작은 샘플보기
import random

random.seed(42)
src,tgt = generate_sample(min_len=3, max_len=5)

print("src(인덱스):", src)
print("tgt(인덱스):", tgt)
print("src(토큰)", [idx_to_token[x] for x in src])
print("tgt(토큰)", [idx_to_token[x] for x in tgt])

src(인덱스): [4, 3, 7, 6, 6]
tgt(인덱스): [6, 6, 7, 3, 4]
src(토큰) ['2', '1', '5', '4', '4']
tgt(토큰) ['4', '4', '5', '1', '2']


In [31]:
def bulid_dataset(n_samples=1000):
    # 샘플 여러 개를 모아서 데이터셋 생성
    data = []
    for _ in range(n_samples):
        src,tgt = generate_sample()
        data.append((src,tgt))
    return data

In [32]:
# 학습용 데이터와 테스트용 데이터 생성
train_data = bulid_dataset(1000)
test_data = bulid_dataset(10)

In [33]:
# 패딩 함수와 배치 생성 함수

# RNN/GRU 여러 문장을 한꺼번에 넣을려면 고정된 길이로 맞춰야함.
# ex) [1,2,3]
##ex) [4,5] -> [4,5,PAD] / 가장 긴 것을 기준으로 PAD를 하다보면 의미가 깨질수도 있고
## 길이가 길어지다 보니 기억력이 떨어지는 경우도 보임.

def pad_sequence(seq, max_len,pad_value=PAD_IDX):
    # 현재 시퀀스 길이가 max_len보다 작으면
    ## 뒤에 PAD 토큰을 붙여서 길이를 맞춰줌.
    return seq + [pad_value] * (max_len - len(seq))

def make_batch(data_batch):
    # 한 배치 안의 src, decoder 입력, target을 담을 리스트
    src_batch = []
    dec_input_batch = []
    target_batch = []

    for src,tgt in data_batch:
        src_seq = src + [EOS_IDX]

        # decoder 입력은 항상 SOS로 시작-> 그 뒤에 정답 시퀀스를 붙임.
        dec_input_seq = [SOS_IDX] + tgt

        # decoder가 맞춰야 할 target은 실제 정답 EOS를 붙인 형태
        target_seq = tgt + [EOS_IDX]

        src_batch.append(src_seq)
        dec_input_batch.append(dec_input_seq)
        target_batch.append(target_seq)
    # 각 배치에서 가장 긴 길이를 찾음.
    src_max_len = max(len(x) for x in src_batch)
    dec_max_len = max(len(x) for x in dec_input_batch)
    tgt_max_len = max(len(x) for x in target_batch)

    # 가장 긴 길이에 맞춰 PAD 추가
    src_batch = [pad_sequence(x,src_max_len) for x in src_batch]
    dec_input_batch = [pad_sequence(x,dec_max_len) for x in dec_input_batch]
    target_batch = [pad_sequence(x,tgt_max_len) for x in target_batch]

    # 파이토치 텐서로 변환 후 device로 이동
    src_tensor = torch.tensor(src_batch, dtype=torch.long, device=device)
    dec_input_tensor = torch.tensor(dec_input_batch, dtype=torch.long, device=device)
    target_tensor = torch.tensor(target_batch, dtype=torch.long, device=device)

    return src_tensor, dec_input_tensor,target_tensor

In [34]:
# make_batch()가 실제로 무엇을 만드는 지?
## decoder 입력과 target이 어떻게 다른지?

sample1 = ([3,4,5],[5,4,3])
sample2 = ([6,7], [7,6])

batch = [sample1,sample2]

src_tensor, dec_input_tensor, target_tensor = make_batch(batch)

print("src_tensor")
print(src_tensor)
print()

print("dec_input_tensor")
print(dec_input_tensor)
print()

print("target_tensor")
print(target_tensor)
print()

src_tensor
tensor([[3, 4, 5, 2],
        [6, 7, 2, 0]], device='cuda:0')

dec_input_tensor
tensor([[1, 5, 4, 3],
        [1, 7, 6, 0]], device='cuda:0')

target_tensor
tensor([[5, 4, 3, 2],
        [7, 6, 2, 0]], device='cuda:0')



In [35]:
def show_tensor_as_tokens(tensor, name):
    print(f"--- {name} ---")
    for row in tensor.tolist():
        print(row, "->", [idx_to_token[x] for x in row])
    print()

show_tensor_as_tokens(src_tensor, "src_tensor")
show_tensor_as_tokens(dec_input_tensor, "dec_input_tensor")
show_tensor_as_tokens(target_tensor, "target_tensor")

--- src_tensor ---
[3, 4, 5, 2] -> ['1', '2', '3', '<EOS>']
[6, 7, 2, 0] -> ['4', '5', '<EOS>', '<PAD>']

--- dec_input_tensor ---
[1, 5, 4, 3] -> ['<SOS>', '3', '2', '1']
[1, 7, 6, 0] -> ['<SOS>', '5', '4', '<PAD>']

--- target_tensor ---
[5, 4, 3, 2] -> ['3', '2', '1', '<EOS>']
[7, 6, 2, 0] -> ['5', '4', '<EOS>', '<PAD>']



In [41]:
# Encoder
## 입력 시퀀스를 읽고 마지막 hidden layer(context vector)에 입력 문장의 정보를 압축하여 담음

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()

        # 단어 인덱스 -> Dense vector(임베딩)로 바꾸는 층
        ## 3 -> [0.1, -0.2,...]
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = emb_dim,
            padding_idx = PAD_IDX
        )
        #GRU 층
        self.gru = nn.GRU(
            input_size = emb_dim,
            hidden_size = hidden_dim,
            batch_first=True)
    def forward(self,src):
        # src : (batch_size,src_len)

        #각 토큰을 임베딩 벡터로 변환.
        ## (embedding shape: (batch_size,src_len, emb_dim)
        embedded = self.embedding(src)

        # GRU 넣어 시퀀스를 순서대로 읽음.
        ## outputs: 각 시점의 hidden_state
        ## hidden : 마지막 시점의 hidden_state

        # Output shape: (batch_size, src_len, hidden_dim)
        # hidden_shape : (1, batch_size, hidden_dim)
        outputs, hidden = self.gru(embedded)

        return outputs,hidden


In [42]:
# Encoder 통과 중간 보기
emb_dim = 8
hidden_dim = 16

encoder = Encoder(vocab_size, emb_dim, hidden_dim).to(device)

src_tensor, dec_input_tensor, target_tensor = make_batch(batch)

print("src_tensor shape:", src_tensor.shape)  # (batch_size, src_len)

embedded = encoder.embedding(src_tensor)
print("embedded shape:", embedded.shape)      # (batch_size, src_len, emb_dim)

outputs, hidden = encoder(src_tensor)
print("encoder outputs shape:", outputs.shape) # (batch_size, src_len, hidden_dim)
print("encoder hidden shape:", hidden.shape)   # (1, batch_size, hidden_dim)

src_tensor shape: torch.Size([2, 4])
embedded shape: torch.Size([2, 4, 8])
encoder outputs shape: torch.Size([2, 4, 16])
encoder hidden shape: torch.Size([1, 2, 16])


In [44]:
# Decoder
## encoder의 hidden_state(context vector)를 받아 출력 시퀀스를 한 단계식 생성
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()

        # decoder 입력 토큰도 임베딩 변환
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = emb_dim,
            padding_idx = PAD_IDX
        )
        #GRU 층
        self.gru = nn.GRU(
            input_size = emb_dim,
            hidden_size = hidden_dim,
            batch_first=True)
        # hidden_state를 단어 분류 점수(Logits-> 0~1로 바꾸는 함수))로 바꾸는 선형층
        # hidden_dim -> vocab_size
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self,dec_input, hidden):
        # dec_input shape: (batch_size, tgt_len)
        # hidden shape : (1, batch_size, hidden_dim)

        # decoder 토큰 임베딩
        ## embedded shape: (batch_size, tgt_len, emb_dim)
        embedded = self.embedding(dec_input)

        # 이전 hidden을 받아 다음 출력 계산
        # output_shape: (batch_size, tgt_len, hidden_dim)
        outputs, hidden = self.gru(embedded, hidden)

        #각 시점마다 어떤 단어가 나와야 하는지 점수 계산
        # logits shape: (batch_size, tgt_len, vocab_size)
        logits = self.fc(outputs)

        return logits, hidden

In [45]:
# Decoder의 중간보기
decoder = Decoder(vocab_size, emb_dim, hidden_dim).to(device)

logits, dec_hidden = decoder(dec_input_tensor, hidden)

print("dec_input_tensor shape:", dec_input_tensor.shape)  # (batch_size, tgt_len)
print("logits shape:", logits.shape)                      # (batch_size, tgt_len, vocab_size)
print("dec_hidden shape:", dec_hidden.shape)              # (1, batch_size, hidden_dim)

dec_input_tensor shape: torch.Size([2, 4])
logits shape: torch.Size([2, 4, 13])
dec_hidden shape: torch.Size([1, 2, 16])


In [48]:
class Seq2Seq(nn.Module):
    def __init__(self,encoder,decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self,src, dec_input):
        # 1. encoder기 입력 문장을 읽음
        _, hidden = self.encoder(src)

        # 2. encoder의 마지막 hidden을 decoder 초기 hidden에 전달
        logits, _ = self.decoder(dec_input,hidden)
        return logits

In [49]:
# 한번만 학습해보고 손실보기
model = Seq2Seq(
    Encoder(vocab_size, emb_dim=8, hidden_dim=16),
    Decoder(vocab_size, emb_dim=8, hidden_dim=16)
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.01)

src_tensor, dec_input_tensor, target_tensor = make_batch(batch)

# 학습 전 예측
model.eval()
with torch.no_grad():
    logits_before = model(src_tensor, dec_input_tensor)
    pred_before = logits_before.argmax(dim=-1)

print("학습 전 예측")
show_tensor_as_tokens(pred_before, "pred_before")
show_tensor_as_tokens(target_tensor, "target")

# 학습 1 step
model.train()
optimizer.zero_grad()

logits = model(src_tensor, dec_input_tensor)

loss = criterion(
    logits.reshape(-1, vocab_size),
    target_tensor.reshape(-1)
)

print("loss:", loss.item())

loss.backward()
optimizer.step()

# 학습 후 예측
model.eval()
with torch.no_grad():
    logits_after = model(src_tensor, dec_input_tensor)
    pred_after = logits_after.argmax(dim=-1)

print("학습 후 예측")
show_tensor_as_tokens(pred_after, "pred_after")
show_tensor_as_tokens(target_tensor, "target")

학습 전 예측
--- pred_before ---
[1, 3, 1, 1] -> ['<SOS>', '1', '<SOS>', '<SOS>']
[1, 1, 1, 1] -> ['<SOS>', '<SOS>', '<SOS>', '<SOS>']

--- target ---
[5, 4, 3, 2] -> ['3', '2', '1', '<EOS>']
[7, 6, 2, 0] -> ['5', '4', '<EOS>', '<PAD>']

loss: 2.532865285873413
학습 후 예측
--- pred_after ---
[1, 3, 3, 3] -> ['<SOS>', '1', '1', '1']
[1, 3, 2, 3] -> ['<SOS>', '1', '<EOS>', '1']

--- target ---
[5, 4, 3, 2] -> ['3', '2', '1', '<EOS>']
[7, 6, 2, 0] -> ['5', '4', '<EOS>', '<PAD>']



In [ ]:
#모델 생성
## emb_dim : 임베딩 벡터 길이
## hidden_dim: GRU Hidden state 크기

emb_dim = 32
hidden_dim = 64

encoder = Encoder(vocab_size, emb_dim, hidden_dim)
decoder = Decoder(vocab_size, emb_dim, hidden_dim)

# 전체 seq2seq 모델 생성 후 device로 이동
model = Seq2Seq(encoder,decoder).to(device)

# CrossEntropyLoss
## 각 시점에서 정답 단어를 맞히는 다중 분류 손실
## ignore_idx = PAX_IDX이므로 PAD 위치는 손실 계산에서 제외
criterion = nn.CrossEntropyLoss(ignore_index = PAD_IDX)

# Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)


# 한 epoch 학습 함수

def train_one_epoch(model, data,batch_size=32):
    #학습 모드 전환
    model.train()

    #데이터 순서를 랜덤하게 섞음.
    random.shuffle(data)

    train_loss = 0

    #batch_size씩 끊어서 반복
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]

        # 배치 벡터 생성
        src, dec_input, target = make_batch(batch)

        # 이전 step의 gradient 초기화
        optimizer.zero_grad()

        # 순전파
        ## logits shape: (batch_size, tgt_len, vocab_size)
        logits = model(src,dec_input)
        
        ## CrossEntropyLoss의 입력이 (N,클래스개수), taget이 (N,)이여야 편함
        ### 3차원 logits이기 때문에 2차원으로 펼치고, taget도 1차원으로 펼침
        
        loss = criterion(
            logits.reshape(-1,vocab_size),
            target.reshape(-1)
        )
        #역전파
        loss.backward()

        #파라미터 업데이트
        optimizer.step()

        total_loss += loss.item()
    return total_loss / (len(data) / batch_size)

In [ ]:
# 추론 함수
## 학습과는 다르게 추론 때, 정답 target이 없음
## decoder는 자기 자신의 이전 예측을 다음 이력으로 사용함.

def predict(model,src_seq, max_len=10):
    #평가 모드로 전환
    model.eval()

    # 입력 시퀀스 끝에 EOS를 붙이고 batch 차원 추가 # shape:(1,src_len)
    src = torch.tensor([src_seq + [EOS_IDX]], dtype=torch.long, device=device)

    with torch.no_grad():
        #encoder 통과
        _, hidden = model.encoder(src)

        # decoder의 첫 입력은 SOS
        dec_input = torch.tensor([[SOS_IDX]], dtype=torch.long, device=device)

        # 예측 결과를 저장할 리스트
        result = []

        for _ in range(max_len):
            #현재 입력 1개를 decoder에 넣어 다음 토큰을 예측
            logits, hidden = model.decoder(dec_input, hidden)

            # 마지막 시점의 예측값 가져오기
            ## shape : (1,vocab_size) -> 가장 큰 점수의 index를 선택
            next_token = logits[:,-1,:].argmax(dim=-1).item()

            # EOS나오면 문장 종료
            if next_token == EOS_IDX:
                break

            # 예측 토큰 저장
            result.append(next_token)

            #방금 예측한 토큰을 다음 decoder 입력으로 사용
            dec_input = torch.tensor([[next_token]], dtype=torch.long, device=device)
        return result

In [ ]:
# 숫자 인덱스를 다시 토큰 문자열로 바꾸는 함수
def decode_token(token_ids):
    return [idx_to_token[idx] for idx in token_ids]
    
    # 학습 실행
for epoch in range(20):
    loss = train_one_epoch(model, train_data, batch_size=32)
    print(f"Epoch {epoch+1:2d}, Loss: {loss:.4f}")
    
    # 테스트
print("테스트 결과")
for src,tgt in test_data:
    pred = predict(model,src)

    print("입력:", decode_token(src))
    print("정답:", decode_token(tgt))
    print("예측:", decode_token(pred))
    print("-"*30)